In [1]:
from datetime import datetime
import json
import codecs
from ast import literal_eval
import random

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


### 1. Leer los datos

In [3]:
df_customers = pd.read_json('/content/drive/MyDrive/Machine Learning/1. Víctor Cordero/00.Introduccion/ejercicio3/customer.json',
                            lines=True,
                            encoding='utf-8-sig')

In [4]:
df_customers

,fraudulent,customer,orders,paymentMethods,transactions
0,False,"{'customerEmail': 'josephhoward@yahoo.com', 'c...","[{'orderId': 'vjbdvd', 'orderAmount': 18, 'ord...","[{'paymentMethodId': 'wt07xm68b', 'paymentMeth...","[{'transactionId': 'a9lcj51r', 'orderId': 'vjb..."
1,True,"{'customerEmail': 'evansjeffery@yahoo.com', 'c...","[{'orderId': 'nlghpa', 'orderAmount': 45, 'ord...","[{'paymentMethodId': 'y3xp697jx', 'paymentMeth...","[{'transactionId': '5mi94sfw', 'orderId': 'nlg..."
2,False,"{'customerEmail': 'andersonwilliam@yahoo.com',...","[{'orderId': 'yk34y2', 'orderAmount': 33, 'ord...","[{'paymentMethodId': '8pneoi03z', 'paymentMeth...","[{'transactionId': 'q3lyvbza', 'orderId': 'yk3..."
3,False,"{'customerEmail': 'rubenjuarez@yahoo.com', 'cu...","[{'orderId': 'fbz9ep', 'orderAmount': 34, 'ord...","[{'paymentMethodId': 'pdxjdwui4', 'paymentMeth...","[{'transactionId': 'vx4cjc27', 'orderId': 'fbz..."
4,True,"{'customerEmail': 'uchen@malone.com', 'custome...","[{'orderId': '56h7iw', 'orderAmount': 71, 'ord...","[{'paymentMethodId': 'w1i1zq3rg', 'paymentMeth...","[{'transactionId': 'q8j3dgni', 'orderId': '56h..."
...,...,...,...,...,...
163,True,{'customerEmail': 'mitchellvickie@brewer-jones...,"[{'orderId': '6086rd', 'orderAmount': 31, 'ord...","[{'paymentMethodId': 'x644o186l', 'paymentMeth...","[{'transactionId': 'ryxa1mzn', 'orderId': '608..."
164,True,"{'customerEmail': 'sbrown@hughes.biz', 'custom...","[{'orderId': '57yhdo', 'orderAmount': 30, 'ord...","[{'paymentMethodId': 'wsy5c41nf', 'paymentMeth...","[{'transactionId': '293g6767', 'orderId': '57y..."
165,True,"{'customerEmail': 'johnlowery@gmail.com', 'cus...","[{'orderId': 'vcyuse', 'orderAmount': 35, 'ord...","[{'paymentMethodId': 'j3foo2l16', 'paymentMeth...","[{'transactionId': 'qio8vvq7', 'orderId': 'vcy..."
166,True,{'customerEmail': 'ethompson@jackson-sanders.c...,"[{'orderId': '9s9ne9', 'orderAmount': 49, 'ord...","[{'paymentMethodId': 'b3jl38c9g', 'paymentMeth...","[{'transactionId': '1uusmhj9', 'orderId': '9s9..."


### 2. Extraer las columnas de los clientes

In [5]:
df_customers['customer'][0]

{'customerEmail': 'josephhoward@yahoo.com',
 'customerPhone': '400-108-5415',
 'customerDevice': 'yyeiaxpltf82440jnb3v',
 'customerIPAddress': '8.129.104.40',
 'customerBillingAddress': '5493 Jones Islands\nBrownside, CA 51896'}

In [6]:
df_customers['customerEmail'] = df_customers['customer'].apply(lambda x:x['customerEmail'])
df_customers['customerPhone'] =df_customers['customer'].apply(lambda x:x['customerPhone'])
df_customers['customerDevice'] = df_customers['customer'].apply(lambda x:x['customerDevice'])
df_customers['customerIPAddress'] = df_customers['customer'].apply(lambda x:x['customerIPAddress'])
df_customers['customerBillingAddress'] = df_customers['customer'].apply(lambda x:x['customerBillingAddress'])

In [7]:
df_customers.drop('customer', axis=1, inplace=True)

In [8]:
df_customers

,fraudulent,orders,paymentMethods,transactions,customerEmail,customerPhone,customerDevice,customerIPAddress,customerBillingAddress
0,False,"[{'orderId': 'vjbdvd', 'orderAmount': 18, 'ord...","[{'paymentMethodId': 'wt07xm68b', 'paymentMeth...","[{'transactionId': 'a9lcj51r', 'orderId': 'vjb...",josephhoward@yahoo.com,400-108-5415,yyeiaxpltf82440jnb3v,8.129.104.40,"5493 Jones Islands\nBrownside, CA 51896"
1,True,"[{'orderId': 'nlghpa', 'orderAmount': 45, 'ord...","[{'paymentMethodId': 'y3xp697jx', 'paymentMeth...","[{'transactionId': '5mi94sfw', 'orderId': 'nlg...",evansjeffery@yahoo.com,1-788-091-7546,r0jpm7xaeqqa3kr6mzum,219.173.211.202,356 Elizabeth Inlet Suite 120\nPort Joshuabury...
2,False,"[{'orderId': 'yk34y2', 'orderAmount': 33, 'ord...","[{'paymentMethodId': '8pneoi03z', 'paymentMeth...","[{'transactionId': 'q3lyvbza', 'orderId': 'yk3...",andersonwilliam@yahoo.com,024.420.0375,4m7h5ipl1shyavt6vv2r,67b7:3db8:67e0:3bea:b9d0:90c1:2b60:b9f0,"8478 Sean Ridges Apt. 441\nDavisberg, PR 72250"
3,False,"[{'orderId': 'fbz9ep', 'orderAmount': 34, 'ord...","[{'paymentMethodId': 'pdxjdwui4', 'paymentMeth...","[{'transactionId': 'vx4cjc27', 'orderId': 'fbz...",rubenjuarez@yahoo.com,670.664.8168x94985,slovx60t0i558may4ks0,95de:8565:5a66:792c:26e0:6cfb:7d87:11af,"7769 Elizabeth Bridge Apt. 343\nNortonstad, FM..."
4,True,"[{'orderId': '56h7iw', 'orderAmount': 71, 'ord...","[{'paymentMethodId': 'w1i1zq3rg', 'paymentMeth...","[{'transactionId': 'q8j3dgni', 'orderId': '56h...",uchen@malone.com,1-981-877-0870,j0pd24k5h8dl2fqu0cz4,196.89.235.192,"148 Russell Lodge Apt. 445\nPort Jenniferside,..."
...,...,...,...,...,...,...,...,...,...
163,True,"[{'orderId': '6086rd', 'orderAmount': 31, 'ord...","[{'paymentMethodId': 'x644o186l', 'paymentMeth...","[{'transactionId': 'ryxa1mzn', 'orderId': '608...",mitchellvickie@brewer-jones.com,313-124-3836x3528,ecu7ezz0ue67v3fce1o3,181.90.104.120,"829 Ramirez Dam Suite 648\nGatesview, OR 55669..."
164,True,"[{'orderId': '57yhdo', 'orderAmount': 30, 'ord...","[{'paymentMethodId': 'wsy5c41nf', 'paymentMeth...","[{'transactionId': '293g6767', 'orderId': '57y...",sbrown@hughes.biz,(250)632-6631x02499,xgv918sclaz40qlp2fee,16.166.2.0,"9472 Pacheco Islands Suite 940\nDrewburgh, AZ ..."
165,True,"[{'orderId': 'vcyuse', 'orderAmount': 35, 'ord...","[{'paymentMethodId': 'j3foo2l16', 'paymentMeth...","[{'transactionId': 'qio8vvq7', 'orderId': 'vcy...",johnlowery@gmail.com,635.676.0955x524,969zz9zdj8z4gns4sx5p,f82c:811f:8a02:e2d6:79b:fcaa:42de:570b,"548 Bryant Inlet\nVeronicaside, OK 00522"
166,True,"[{'orderId': '9s9ne9', 'orderAmount': 49, 'ord...","[{'paymentMethodId': 'b3jl38c9g', 'paymentMeth...","[{'transactionId': '1uusmhj9', 'orderId': '9s9...",ethompson@jackson-sanders.com,962.300.2250,navessqcnmc9csfjn5cj,203.246.172.244,"16953 Jennifer Springs\nKathybury, IL 02095-3215"


In [9]:
df_customers['orders'].loc[0]

[{'orderId': 'vjbdvd',
  'orderAmount': 18,
  'orderState': 'pending',
  'orderShippingAddress': '5493 Jones Islands\nBrownside, CA 51896'},
 {'orderId': 'yp6x27',
  'orderAmount': 26,
  'orderState': 'fulfilled',
  'orderShippingAddress': '5493 Jones Islands\nBrownside, CA 51896'}]

### 3. Generar los dataframes , "aplanando los diccionarios" y generando columnas nuevas para cada cliente

#### Dataframe orders

In [10]:
df_orders = df_customers[['orders', 'customerPhone']]

In [11]:
df_orders.shape

(168, 2)

In [12]:
df_orders.explode('orders')

,orders,customerPhone
0,"{'orderId': 'vjbdvd', 'orderAmount': 18, 'orde...",400-108-5415
0,"{'orderId': 'yp6x27', 'orderAmount': 26, 'orde...",400-108-5415
1,"{'orderId': 'nlghpa', 'orderAmount': 45, 'orde...",1-788-091-7546
1,"{'orderId': 'uw0eeb', 'orderAmount': 23, 'orde...",1-788-091-7546
1,"{'orderId': 'bn44oh', 'orderAmount': 43, 'orde...",1-788-091-7546
...,...,...
167,"{'orderId': 'ow1cih', 'orderAmount': 23, 'orde...",(089)703-7360x8848
167,"{'orderId': 'ak5a9n', 'orderAmount': 32, 'orde...",(089)703-7360x8848
167,"{'orderId': '1bubxa', 'orderAmount': 25, 'orde...",(089)703-7360x8848
167,"{'orderId': 'p0gdbf', 'orderAmount': 19, 'orde...",(089)703-7360x8848


In [13]:
df_orders.explode('orders')

,orders,customerPhone
0,"{'orderId': 'vjbdvd', 'orderAmount': 18, 'orde...",400-108-5415
0,"{'orderId': 'yp6x27', 'orderAmount': 26, 'orde...",400-108-5415
1,"{'orderId': 'nlghpa', 'orderAmount': 45, 'orde...",1-788-091-7546
1,"{'orderId': 'uw0eeb', 'orderAmount': 23, 'orde...",1-788-091-7546
1,"{'orderId': 'bn44oh', 'orderAmount': 43, 'orde...",1-788-091-7546
...,...,...
167,"{'orderId': 'ow1cih', 'orderAmount': 23, 'orde...",(089)703-7360x8848
167,"{'orderId': 'ak5a9n', 'orderAmount': 32, 'orde...",(089)703-7360x8848
167,"{'orderId': '1bubxa', 'orderAmount': 25, 'orde...",(089)703-7360x8848
167,"{'orderId': 'p0gdbf', 'orderAmount': 19, 'orde...",(089)703-7360x8848


In [14]:
df_orders['orders'].isnull().sum()

np.int64(0)

In [15]:
df_orders = df_orders[df_orders['orders'].notnull()]

In [17]:
df_orders['orders'].apply(lambda x:x['orderId'])

TypeError: list indices must be integers or slices, not str

,orders
0,"[{'orderId': 'vjbdvd', 'orderAmount': 18, 'ord..."
1,"[{'orderId': 'nlghpa', 'orderAmount': 45, 'ord..."
2,"[{'orderId': 'yk34y2', 'orderAmount': 33, 'ord..."
3,"[{'orderId': 'fbz9ep', 'orderAmount': 34, 'ord..."
4,"[{'orderId': '56h7iw', 'orderAmount': 71, 'ord..."
...,...
163,"[{'orderId': '6086rd', 'orderAmount': 31, 'ord..."
164,"[{'orderId': '57yhdo', 'orderAmount': 30, 'ord..."
165,"[{'orderId': 'vcyuse', 'orderAmount': 35, 'ord..."
166,"[{'orderId': '9s9ne9', 'orderAmount': 49, 'ord..."


In [ ]:
df_customers['orders'][1]

[{'orderId': 'nlghpa',
  'orderAmount': 45,
  'orderState': 'fulfilled',
  'orderShippingAddress': '898 Henry Ports\nNew Keithview, CA 95893-2497'},
 {'orderId': 'uw0eeb',
  'orderAmount': 23,
  'orderState': 'fulfilled',
  'orderShippingAddress': '356 Elizabeth Inlet Suite 120\nPort Joshuabury, NM 37681'},
 {'orderId': 'bn44oh',
  'orderAmount': 43,
  'orderState': 'fulfilled',
  'orderShippingAddress': '5093 Bryan Forks\nJoshuaton, FM 01565-9801'}]

#### Dataframe payment methods

#### Dataframe transactions

### 4. Unir los dataframes en un tablón analítico único final

### 5. Trabajar con el tablón

#### Obtener los tipos de datos

#### Sacar el número de nulos y trabajar con ellos

#### Seleccionar columnas finales

#### Visualización